
# <font color='blue'>Eduardo Lyrio</font>
## <font color='blue'>Pipelines de Análise e Engenharia de Dados com Google BigQuery</font>
## <font color='blue'>Projeto 3</font>
### <font color='blue'>Pipeline de Análise de Cesta de Compras (Market Basket Analysis) com Python e Google BigQuery</font>

## Instalando e Carregando os Pacotes

In [1]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.2 MB/s eta 0:00:00


In [2]:
!pip install -q google-cloud-bigquery

In [3]:
!pip install -q db-dtypes

In [4]:
!pip install -q mlxtend

In [5]:
# Imports
import os
import mlxtend
import pandas as pd
from google.cloud import bigquery
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

In [6]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Eduardo Lyrio"

Author: Eduardo Lyrio



## Conectando ao Google BigQuery

In [13]:
# Define o caminho do arquivo de credenciais
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "capitulo08dsa-489811-6c9ead5774ee.json"

In [14]:
# Instancia o cliente do BigQuery
client = bigquery.Client()

In [15]:
# Verifica se a autenticação foi feita corretamente
print("Conectado ao BigQuery")

Conectado ao BigQuery


## Extração dos Dados de Transações

In [16]:
# Query SQL (ajuste o id do projeto e nome do dataset)
query = """
    SELECT
        f.Cliente_ID,
        f.Produto_ID,
        p.Nome AS Produto,
        f.Quantidade
    FROM
        `capitulo08dsa-489811.projeto2.fato_venda` f
    JOIN
        `capitulo08dsa-489811.projeto2.dim_produto` p
    ON
        f.Produto_ID = p.Produto_ID
"""

In [17]:
# Executa a consulta e armazena os resultados em um dataframe
df = client.query(query).to_dataframe()

In [18]:
df.head()

,Cliente_ID,Produto_ID,Produto,Quantidade
0,8,2,Produto B,14
1,10,3,Produto C,3
2,5,4,Produto D,7
3,1,7,Produto G,3
4,7,7,Produto G,18


In [19]:
df.shape

(50, 4)

## Cria Matriz de Transações Cliente x Produto x Quantidade

In [20]:
# Cria uma matriz de transações Cliente x Produto x Quantidade
basket = df.pivot_table(index = 'Cliente_ID',
                        columns = 'Produto',
                        values = 'Quantidade',
                        aggfunc = 'sum').fillna(0)

In [21]:
basket

Produto,Produto A,Produto B,Produto C,Produto D,Produto E,Produto F,Produto G,Produto H,Produto I,Produto J
Cliente_ID,,,,,,,,,,
1,4,16,0,35,0,0,3,17,0,3
2,0,0,17,0,1,17,0,0,17,2
3,0,15,0,0,17,0,0,0,5,0
4,0,0,8,0,6,0,0,0,18,0
5,0,0,6,7,14,5,0,0,0,18
6,12,2,12,0,10,0,1,1,8,0
7,0,0,0,0,1,2,18,0,0,14
8,0,24,0,0,19,0,19,0,0,16
9,0,0,0,0,0,0,16,14,0,0


In [22]:
# Converte valores para binário: 1 para quantidade > 0 e 0 para nenhum produto na cesta
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

In [23]:
basket

Produto,Produto A,Produto B,Produto C,Produto D,Produto E,Produto F,Produto G,Produto H,Produto I,Produto J
Cliente_ID,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,1
2,0,0,1,0,1,1,0,0,1,1
3,0,1,0,0,1,0,0,0,1,0
4,0,0,1,0,1,0,0,0,1,0
5,0,0,1,1,1,1,0,0,0,1
6,1,1,1,0,1,0,1,1,1,0
7,0,0,0,0,1,1,1,0,0,1
8,0,1,0,0,1,0,1,0,0,1
9,0,0,0,0,0,0,1,1,0,0


## Algoritmo Apriori

Leia os detalhes sobre o algoritmo apriori no videobook do Capítulo 9 do curso.

O Algoritmo Apriori é um método utilizado em mineração de dados para encontrar associações frequentes em grandes bases de dados, especialmente útil em análises de cestas de compras. Seu objetivo é identificar padrões recorrentes e regras de associação que indicam a probabilidade de itens serem comprados juntos.

In [24]:
# Aplica o algoritmo
frequent_itemsets = apriori(basket, min_support = 0.3, use_colnames = True)

## Regras de Associação

Leia os detalhes sobre regras de associação no videobook do Capítulo 9 do curso.

A extração de Regras de Associação a partir dos conjuntos frequentes (frequent itemsets) gerados pelo Algoritmo Apriori envolve a identificação de relações significativas entre os itens presentes nesses conjuntos.

In [25]:
# Gerando regras de associação
rules = association_rules(frequent_itemsets, metric = "confidence", min_threshold = 0.6)

In [26]:
# Ordenando as regras por lift
rules = rules.sort_values('lift', ascending = False)

In [27]:
rules.head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
22,(Produto F),"(Produto J, Produto E)",0.3,0.4,0.3,1.00,2.5,1.0,0.18,inf,0.857143,0.75,1.000000,0.875
19,"(Produto J, Produto E)",(Produto F),0.4,0.3,0.3,0.75,2.5,1.0,0.18,2.80,1.000000,0.75,0.642857,0.875
8,(Produto J),(Produto F),0.5,0.3,0.3,0.60,2.0,1.0,0.15,1.75,1.000000,0.60,0.428571,0.800
9,(Produto F),(Produto J),0.3,0.5,0.3,1.00,2.0,1.0,0.15,inf,0.714286,0.60,1.000000,0.800
20,"(Produto F, Produto E)",(Produto J),0.3,0.5,0.3,1.00,2.0,1.0,0.15,inf,0.714286,0.60,1.000000,0.800


Aqui está uma interpretação detalhada das métricas para as regras de associação listadas:

**Antecedents (Antecedentes)**: Os itens ou conjunto de itens no lado esquerdo da regra. Eles representam a condição inicial da associação.

**Consequents (Consequentes)**: Os itens ou conjunto de itens no lado direito da regra. Eles são o resultado esperado quando os antecedentes ocorrem.

**Antecedent Support (Suporte do Antecedente)**: A frequência relativa com que o antecedente ocorre no conjunto de dados. Por exemplo, se o antecedente é o "Produto F" com um suporte de 0.3, significa que o "Produto F" aparece em 30% de todas as transações.

**Consequent Support (Suporte do Consequente)**: A frequência relativa com que o consequente ocorre no conjunto de dados. Para "Produto J, Produto E" com um suporte de 0.4, significa que ambos aparecem juntos em 40% das transações.

**Support (Suporte)**: A frequência com que ambos, o antecedente e o consequente, ocorrem juntos nas transações. Por exemplo, um suporte de 0.3 indica que 30% das transações contêm tanto o antecedente quanto o consequente.

**Confidence (Confiança)**: Mede a probabilidade de o consequente ocorrer quando o antecedente ocorre. É calculado como a razão entre o suporte da regra e o suporte do antecedente. Uma confiança de 1.0 indica que toda vez que o antecedente aparece, o consequente também aparece.

**Lift (Efeito ou Elevação)**: Avalia a força da associação entre antecedente e consequente comparando com o que seria esperado se eles fossem independentes. Um lift de 2.5 significa que o antecedente aumenta a chance de o consequente ocorrer em 2.5 vezes, sugerindo uma associação positiva.

**Leverage**: Mede o quão mais frequentemente o antecedente e o consequente ocorrem juntos do que o esperado se fossem independentes. Um valor positivo indica uma associação útil; por exemplo, um leverage de 0.18 sugere uma associação além da independência.

**Conviction**: Mede a independência das ocorrências de antecedentes e consequentes, comparando a frequência real de ocorrência do antecedente sem o consequente com a frequência esperada. Um valor alto indica uma relação de dependência forte. Valores de "inf" indicam que a consequente praticamente nunca ocorre sem o antecedente.

**Zhang's Metric**: Avalia a força da associação em termos de independência, sendo mais robusta contra o viés do suporte. Um valor próximo de 1 indica uma forte relação positiva, enquanto valores mais baixos indicam uma associação fraca. Aqui, valores como 0.857 e 1.0 mostram que há uma associação robusta.

In [28]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Eduardo Lyrio"

Author: Eduardo Lyrio



In [29]:
#!pip show google-cloud-bigquery | grep Version

In [30]:
#!pip show db-dtypes | grep Version

In [31]:
#%watermark -v -m

In [32]:
#%watermark --iversions

# Fim